"""
Replication of Bereta & Diamantis (2025), "The Shape of Consumer Behavior:
A Symbolic and Topological Analysis of Time Series", adapted for 150
Google Trends root-word keywords stored as daily series.
 
Pipeline stages
----------------
1.  Data loading        - reads gt_stitched_fixed_*.csv for every term folder
2.  KS test              - Kolmogorov-Smirnov test for normality (paper Table 3)
3.  SAX                  - sliding-window Symbolic Aggregate approXimation
4.  eSAX                 - Extended SAX (mean/max/min per segment)
5.  TDA                  - sliding-window embedding -> Vietoris-Rips ->
                           persistent homology -> persistence landscapes
6.  Clustering           - KMeans + Hierarchical (Ward) for each representation,
                           with Silhouette and Davies-Bouldin scores
7.  Output               - CSVs + plots replicating the paper's tables/figures
 
Scaling notes (daily vs weekly data)
-------------------------------------
The original paper used WEEKLY Google Trends data (262 points/series) with:
  - SAX/eSAX sliding window  = 52 points  (~1 year)
  - SAX/eSAX segments/window = 12         (~1 month each)
  - TDA embedding            = d=6, tau=3 (~3 weeks lag)
 
Your data is DAILY. To keep the same real-world time spans, all window/lag
parameters are rescaled proportionally (7x, since 1 week = 7 days):
  - SAX/eSAX sliding window  = 365 points (~1 year)   [52 * 7 = 364 ~ 365]
  - SAX/eSAX segments/window = 12         (unchanged - still ~1 month each,
                                            since 365/12 ~= 30.4 days/segment)
  - TDA embedding            = d=6, tau=21 (~3 weeks, i.e. 3*7 days)
 
These are exposed as constants below (WINDOW_SIZE, N_SEGMENTS, EMBED_DIM,
EMBED_TAU) so you can change them in one place if you want the exact
paper values instead, or your own custom values.
"""

In [46]:
import warnings
from pathlib import Path
from dataclasses import dataclass, field
 
import numpy as np
import pandas as pd
import matplotlib
 
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder
from numpy.lib.stride_tricks import sliding_window_view
 
warnings.filterwarnings("ignore")
 
# ----------------------------------------------------------------------------
# CONFIG -- edit these for your environment
# ----------------------------------------------------------------------------
 
DATA_DIR = Path(r"C:\Python\CSUREMM\data_primitive")
OUTPUT_DIR = Path("C://Python//CSUREMM//tda")
 
STITCHED_FILENAME = "gt_stitched_fixed_2022-01-01_2026-05-31.csv"
 
# --- SAX / eSAX parameters (rescaled for daily data; see module docstring) ---
WINDOW_SIZE = 365     # sliding window length, in data points (paper: 52 weekly)
N_SEGMENTS = 12        # PAA segments per window (paper: 12)
ALPHABET_SIZE = 4      # number of SAX symbols (paper uses 4: a,b,c,d)
SAX_STEP = 7            # slide step in points (paper: 7)
BREAKPOINTS = gaussian_breakpoints(ALPHABET_SIZE)
 
# --- TDA parameters (rescaled for daily data; see module docstring) ---
EMBED_DIM = 6           # embedding dimension d (paper: 6)
EMBED_TAU = 21          # time delay tau, in data points (paper: 3 weekly -> 21 daily)
MAX_HOMOLOGY_DIM = 1    # compute H0 and H1
N_LANDSCAPE_POINTS = 200  # discretization resolution for persistence landscapes
N_LANDSCAPE_LAYERS = 5    # number of landscape layers (k=1..5) to keep
 
# --- Clustering ---
N_CLUSTERS = 6          # paper used k=6 throughout (elbow method)
RANDOM_STATE = 42
 
ALPHABET = list("abcdefghijklmnopqrstuvwxyz"[:ALPHABET_SIZE])

In [47]:
# ----------------------------------------------------------------------------
# 1. DATA LOADING
# ----------------------------------------------------------------------------
 
def load_all_series(data_dir: Path, filename: str = STITCHED_FILENAME) -> dict[str, pd.Series]:
    """
    Walk every term subfolder in data_dir, read its stitched daily csv,
    and return {term_name: pandas Series indexed by date}.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []
 
    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue
        fpath = folder / filename
        if not fpath.exists():
            failed.append((folder.name, f"missing file {filename}"))
            continue
        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_col = [c for c in df.columns if c != "Time"][0]
            ts = (
                df.set_index("Time")[value_col]
                .sort_index()
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts
        except Exception as e:
            failed.append((folder.name, str(e)))
 
    if failed:
        print(f"[load_all_series] {len(failed)} terms failed to load:")
        for name, err in failed:
            print(f"    {name}: {err}")
    print(f"[load_all_series] Loaded {len(series_dict)} term series.")
    return series_dict
 
 
def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all series on their shared date index (outer join), as in paper Fig.5/Table 1."""
    panel = pd.DataFrame(series_dict)
    return panel
 
 
def summary_statistics(panel: pd.DataFrame) -> pd.DataFrame:
    """Replicates paper Table 1 (count, mean, std, min, 25/50/75pct, max) for every term."""
    desc = panel.describe(percentiles=[0.25, 0.5, 0.75]).T
    desc = desc.rename(columns={
        "count": "Count", "mean": "Mean", "std": "Std", "min": "Min",
        "25%": "25%", "50%": "50%", "75%": "75%", "max": "Max",
    })
    return desc[["Count", "Mean", "Std", "Min", "25%", "50%", "75%", "Max"]]

In [48]:
# ----------------------------------------------------------------------------
# 2. KOLMOGOROV-SMIRNOV TEST FOR NORMALITY (paper Table 3 / Section 3)
# ----------------------------------------------------------------------------
 
def ks_normality_test(panel: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    For each term's full raw series, run a one-sample KS test against a
    normal distribution fitted to that series' own mean/std (matching the
    paper's approach in Section 3 / Table 3).
    """
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        mu, sigma = x.mean(), x.std(ddof=1)
        if sigma == 0:
            stat, p = np.nan, np.nan
        else:
            stat, p = stats.kstest(x, "norm", args=(mu, sigma))
        rows.append({
            "Keyword": term,
            "KS Statistic": stat,
            "p-value": p,
            "Reject H0 at 5%": "Yes" if (p is not np.nan and p < alpha) else "No",
        })
    out = pd.DataFrame(rows).sort_values("KS Statistic", ascending=False).reset_index(drop=True)
    return out

In [49]:
# ----------------------------------------------------------------------------
# 3. SAX  (sliding-window Symbolic Aggregate approXimation)
# ----------------------------------------------------------------------------
 
BREAKPOINTS = gaussian_breakpoints(ALPHABET_SIZE)

PAA_IDX = np.linspace(
    0,
    WINDOW_SIZE,
    N_SEGMENTS + 1,
    dtype=int
)

LETTER_TO_INT = {c: i for i, c in enumerate(ALPHABET)}


def znormalize(x: np.ndarray) -> np.ndarray:
    mu = np.nanmean(x)
    sigma = np.nanstd(x)

    if sigma == 0 or np.isnan(sigma):
        return np.zeros_like(x)

    return (x - mu) / sigma

def paa(x: np.ndarray) -> np.ndarray:
    return np.array([
        x[PAA_IDX[i]:PAA_IDX[i + 1]].mean()
        for i in range(N_SEGMENTS)
    ])


def value_to_symbol(value: float, breakpoints: np.ndarray) -> str:
    idx = np.searchsorted(breakpoints, value)
    return ALPHABET[idx]


def sax_word(window: np.ndarray) -> str:
    """
    Compute one SAX word.

    ### CHANGE:
    - uses global BREAKPOINTS
    - uses optimized PAA
    """

    z = znormalize(window)
    seg_means = paa(z)

    return "".join(
        value_to_symbol(v, BREAKPOINTS)
        for v in seg_means
    )

def compute_sax_words(
    series: pd.Series,
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = 1
) -> list[str]:

    x = series.values.astype(float)

    if len(x) < window_size:
        return []

    windows = sliding_window_view(x, window_size)[::step]

    ### CHANGE
    means = windows.mean(axis=1, keepdims=True)

    ### CHANGE
    stds = windows.std(axis=1, keepdims=True)

    stds[stds == 0] = 1

    ### CHANGE
    windows = (windows - means) / stds

    words = []

    for w in windows:

        seg_means = paa(w)

        word = "".join(
            value_to_symbol(v, BREAKPOINTS)
            for v in seg_means
        )

        words.append(word)

    return words

def sax_words_to_numeric_matrix(words: list[str]) -> np.ndarray:

    return np.array(
        [
            [LETTER_TO_INT[ch] for ch in w]
            for w in words
        ],
        dtype=np.uint8
    )


def build_sax_feature_matrix(
    series_dict,
    window_size,
    n_segments,
    alphabet_size,
    step=1,
):

    feature_rows = {}

    for term, s in series_dict.items():

        x = s.values.astype(float)

        if len(x) < window_size:
            continue

        ### CHANGE
        windows = sliding_window_view(x, window_size)[::step]

        ### CHANGE
        means = windows.mean(axis=1, keepdims=True)

        stds = windows.std(axis=1, keepdims=True)

        stds[stds == 0] = 1

        windows = (windows - means) / stds

        ### CHANGE
        numeric_words = []

        for w in windows:

            seg_means = paa(w)

            numeric_words.append(
                np.searchsorted(
                    BREAKPOINTS,
                    seg_means
                )
            )

        numeric_words = np.asarray(
            numeric_words,
            dtype=np.uint8
        )

        feature_rows[term] = numeric_words.mean(axis=0)

    feat_df = pd.DataFrame(feature_rows).T

    feat_df.columns = [
        f"seg_{i+1}"
        for i in range(N_SEGMENTS)
    ]

    return feat_df, None

In [50]:
# ----------------------------------------------------------------------------
# 4. eSAX (Extended SAX: mean, max, min per segment, ordered by position)
# ----------------------------------------------------------------------------
 
def esax_word(window: np.ndarray, n_segments: int, breakpoints: np.ndarray) -> str:
    z = znormalize(window)
    n = len(z)
    idx = np.linspace(0, n, n_segments + 1).astype(int)
    letters = []
    for i in range(n_segments):
        seg = z[idx[i]:idx[i + 1]]
        if len(seg) == 0:
            continue
        mean_v, max_v, min_v = seg.mean(), seg.max(), seg.min()
        # order mean/max/min by their position in time within the segment
        positions = {
            "mean": np.argmin(np.abs(seg - mean_v)),
            "max": np.argmax(seg),
            "min": np.argmin(seg),
        }
        ordered = sorted(positions.items(), key=lambda kv: kv[1])
        for key, _ in ordered:
            val = {"mean": mean_v, "max": max_v, "min": min_v}[key]
            letters.append(value_to_symbol(val, breakpoints))
    return "".join(letters)
 
 
def compute_esax_words(
    series: pd.Series,
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = 1,
) -> list[str]:
    """
    Optimized eSAX.

    ### CHANGE
    - sliding_window_view instead of Python slicing
    - vectorized normalization
    - precomputed BREAKPOINTS
    """

    x = series.values.astype(float)

    if len(x) < window_size:
        return []

    ### CHANGE
    windows = sliding_window_view(
        x,
        window_size
    )[::step]

    ### CHANGE
    means = windows.mean(axis=1, keepdims=True)
    stds = windows.std(axis=1, keepdims=True)

    stds[stds == 0] = 1

    windows = (windows - means) / stds

    words = []

    for window in windows:
        words.append(
            esax_word(
                window,
                n_segments,
                BREAKPOINTS
            )
        )

    return words
 
 
LETTER_TO_INT = {
    c: i
    for i, c in enumerate(ALPHABET)
}


def build_esax_feature_matrix(
    series_dict,
    window_size,
    n_segments,
    alphabet_size,
    step=1,
):

    feature_rows = {}
    all_words = {}

    expected_len = n_segments * 3

    for term, s in series_dict.items():

        words = compute_esax_words(
            s,
            window_size,
            n_segments,
            alphabet_size,
            step,
        )

        if not words:
            continue

        words = [
            w
            for w in words
            if len(w) == expected_len
        ]

        if not words:
            continue

        all_words[term] = words

        ### CHANGE
        numeric = np.array(
            [
                [LETTER_TO_INT[ch] for ch in w]
                for w in words
            ],
            dtype=np.uint8,
        )

        feature_rows[term] = numeric.mean(axis=0)

    feat_df = pd.DataFrame(feature_rows).T

    feat_df.columns = [
        f"pos_{i+1}"
        for i in range(expected_len)
    ]

    return feat_df, all_words

In [51]:
# ----------------------------------------------------------------------------
# 5. TDA (Optimized)
# ----------------------------------------------------------------------------

from numpy.lib.stride_tricks import sliding_window_view
from joblib import Parallel, delayed


# ============================================================================
# Sliding Window Embedding
# ============================================================================

def sliding_window_embedding(
    x: np.ndarray,
    dim: int,
    tau: int,
) -> np.ndarray:
    """
    Vectorized Takens embedding.

    ### CHANGE
    Uses sliding_window_view instead of Python loops.
    """

    span = (dim - 1) * tau + 1

    if len(x) < span:
        raise ValueError("Series too short.")

    windows = sliding_window_view(x, span)

    return windows[:, ::tau]


# ============================================================================
# Persistent Homology
# ============================================================================

def compute_persistence_diagrams(
    point_cloud: np.ndarray,
    max_dim: int = 1,
):
    """
    Compute Vietoris-Rips persistence diagrams.
    """
    from ripser import ripser

    result = ripser(
        point_cloud,
        maxdim=max_dim,
    )

    return result["dgms"]


# ============================================================================
# Persistence Landscape
# ============================================================================

def tent_function(
    t: np.ndarray,
    birth: float,
    death: float,
) -> np.ndarray:

    mid = (birth + death) / 2.0
    height = (death - birth) / 2.0

    if height <= 0:
        return np.zeros_like(t)

    val = height - np.abs(t - mid)

    return np.clip(val, 0, None)


def persistence_landscape(
    diagram: np.ndarray,
    n_points: int = N_LANDSCAPE_POINTS,
    n_layers: int = N_LANDSCAPE_LAYERS,
    t_range: tuple[float, float] | None = None,
):

    if len(diagram):
        diagram = diagram[np.isfinite(diagram).all(axis=1)]

    if len(diagram) == 0:
        return np.zeros((n_layers, n_points))

    if t_range is None:
        t_min = diagram[:, 0].min()
        t_max = diagram[:, 1].max()
    else:
        t_min, t_max = t_range

    if t_max <= t_min:
        t_max = t_min + 1

    t = np.linspace(
        t_min,
        t_max,
        n_points,
    )

    tents = np.array(
        [
            tent_function(t, b, d)
            for b, d in diagram
        ]
    )

    if tents.size == 0:
        return np.zeros((n_layers, n_points))

    sorted_tents = -np.sort(
        -tents,
        axis=0,
    )

    landscape = np.zeros(
        (n_layers, n_points)
    )

    k = min(
        n_layers,
        sorted_tents.shape[0],
    )

    landscape[:k] = sorted_tents[:k]

    return landscape


def landscape_to_vector(
    landscape: np.ndarray,
) -> np.ndarray:

    return landscape.ravel()


# ============================================================================
# Parallel Worker
# ============================================================================

def _process_single_tda(
    term,
    series,
    embed_dim,
    embed_tau,
    max_dim,
    n_landscape_points,
    n_layers,
):

    x = series.values.astype(float)

    ### CHANGE
    ### Standardize once before embedding
    mu = np.nanmean(x)
    sigma = np.nanstd(x)

    if sigma == 0 or np.isnan(sigma):
        x = np.zeros_like(x)
    else:
        x = (x - mu) / sigma

    cloud = sliding_window_embedding(
        x,
        embed_dim,
        embed_tau,
    )

    dgms = compute_persistence_diagrams(
        cloud,
        max_dim=max_dim,
    )

    h1_diagram = (
        dgms[1]
        if len(dgms) > 1
        else np.empty((0, 2))
    )

    h0_diagram = (
        dgms[0]
        if len(dgms) > 0
        else np.empty((0, 2))
    )

    h1_landscape = persistence_landscape(
        h1_diagram,
        n_landscape_points,
        n_layers,
    )

    combined = (
        np.vstack(
            [
                h0_diagram,
                h1_diagram,
            ]
        )
        if len(h0_diagram) or len(h1_diagram)
        else np.empty((0, 2))
    )

    h01_landscape = persistence_landscape(
        combined,
        n_landscape_points,
        n_layers,
    )

    return (
        term,
        landscape_to_vector(h1_landscape),
        landscape_to_vector(h01_landscape),
        dgms,
    )


# ============================================================================
# Build TDA Features
# ============================================================================

def build_tda_features(
    series_dict: dict[str, pd.Series],
    embed_dim: int,
    embed_tau: int,
    max_dim: int = MAX_HOMOLOGY_DIM,
    n_landscape_points: int = N_LANDSCAPE_POINTS,
    n_layers: int = N_LANDSCAPE_LAYERS,
):
    """
    Optimized TDA feature extraction.

    ### CHANGE
    - Parallel processing over keywords
    - Vectorized embedding
    - One-time standardization
    - Same outputs as original implementation
    """

    results = Parallel(
        n_jobs=-1,
        prefer="threads"
    )(
        delayed(_process_single_tda)(
            term,
            series,
            embed_dim,
            embed_tau,
            max_dim,
            n_landscape_points,
            n_layers,
        )
        for term, series in series_dict.items()
    )

    h1_rows = {}
    h01_rows = {}
    diagrams_raw = {}

    for result in results:

        if result is None:
            continue

        term, h1_vec, h01_vec, dgms = result

        h1_rows[term] = h1_vec
        h01_rows[term] = h01_vec
        diagrams_raw[term] = dgms

    h1_features = pd.DataFrame.from_dict(
        h1_rows,
        orient="index",
    )

    h01_features = pd.DataFrame.from_dict(
        h01_rows,
        orient="index",
    )

    return (
        h1_features,
        h01_features,
        diagrams_raw,
    )

In [52]:
# ----------------------------------------------------------------------------
# 6. CLUSTERING  (KMeans + Hierarchical/Ward, with Silhouette & DBI)
# ----------------------------------------------------------------------------
 
@dataclass
class ClusterResult:
    method: str               # "KMeans" or "Hierarchical"
    representation: str        # "SAX", "eSAX", "TDA-H1", "TDA-H0H1"
    labels: pd.Series = field(default_factory=pd.Series)
    silhouette: float = np.nan
    davies_bouldin: float = np.nan
 
 
def run_kmeans(features: pd.DataFrame, n_clusters: int = N_CLUSTERS,
                random_state: int = RANDOM_STATE) -> tuple[pd.Series, float, float]:
    X = features.values
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan
    return pd.Series(labels, index=features.index, name="cluster"), sil, dbi
 
 
def run_hierarchical(features: pd.DataFrame, n_clusters: int = N_CLUSTERS,
                      method: str = "ward") -> tuple[pd.Series, float, float, np.ndarray]:
    X = features.values
    Z = linkage(X, method=method)
    labels = fcluster(Z, t=n_clusters, criterion="maxclust") - 1  # zero-index
    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan
    return pd.Series(labels, index=features.index, name="cluster"), sil, dbi, Z
 
 
def elbow_inertias(features: pd.DataFrame, k_range=range(2, 10),
                    random_state: int = RANDOM_STATE) -> pd.Series:
    inertias = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(features.values)
        inertias[k] = km.inertia_
    return pd.Series(inertias)

In [53]:
# ----------------------------------------------------------------------------
# 7. PLOTTING HELPERS
# ----------------------------------------------------------------------------
 
def plot_elbow(inertias: pd.Series, title: str, outpath: Path):
    plt.figure(figsize=(6, 4))
    plt.plot(inertias.index, inertias.values, marker="o")
    plt.xlabel("k (number of clusters)")
    plt.ylabel("Inertia")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()
 
 
def plot_clusters_by_series(series_dict: dict[str, pd.Series], labels: pd.Series,
                             title: str, outpath: Path, n_clusters: int = N_CLUSTERS):
    fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 2.2 * n_clusters), sharex=False)
    if n_clusters == 1:
        axes = [axes]
    for c in range(n_clusters):
        ax = axes[c]
        members = labels[labels == c].index
        for m in members:
            if m in series_dict:
                ax.plot(series_dict[m].index, series_dict[m].values, label=m, linewidth=0.9)
        ax.set_title(f"Cluster {c + 1} ({len(members)} terms)", fontsize=9)
        ax.legend(fontsize=6, loc="upper left", ncol=4)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()
 
 
def plot_dendrogram(Z: np.ndarray, labels: list[str], title: str, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=6)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()
 
 
def plot_ks_bar(ks_df: pd.DataFrame, outpath: Path):
    df = ks_df.sort_values("KS Statistic")
    colors = ["tab:red" if r == "Yes" else "tab:green" for r in df["Reject H0 at 5%"]]
    plt.figure(figsize=(10, max(6, 0.18 * len(df))))
    plt.barh(df["Keyword"], df["KS Statistic"], color=colors)
    plt.xlabel("KS Statistic")
    plt.title("KS Test for Normality per Keyword (red = reject H0 at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


In [ ]:
# ----------------------------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------------------------

# Run time as of June 26: 11 minutes 18.5 seconds on my laptop (i7-9750H, 16GB RAM, 6 cores, 12 threads) for the full 150 root words
 
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
    # ---- 1. Load data ----
    print("=" * 70)
    print("STEP 1: Loading data")
    print("=" * 70)
    series_dict = load_all_series(DATA_DIR, STITCHED_FILENAME)
    panel = build_panel(series_dict)
    summary_statistics(panel).to_csv(OUTPUT_DIR / "table1_summary_statistics.csv")
 
    # ---- 2. KS test ----
    print("\n" + "=" * 70)
    print("STEP 2: Kolmogorov-Smirnov normality test")
    print("=" * 70)
    ks_df = ks_normality_test(panel)
    ks_df.to_csv(OUTPUT_DIR / "table3_ks_test_results.csv", index=False)
    plot_ks_bar(ks_df, OUTPUT_DIR / "fig_ks_test.png")
    print(ks_df.head(10).to_string(index=False))
 
    # ---- 3. SAX ----
    print("\n" + "=" * 70)
    print("STEP 3: SAX (sliding-window symbolic aggregate approximation)")
    print("=" * 70)
    sax_features, sax_words = build_sax_feature_matrix(
        series_dict, WINDOW_SIZE, N_SEGMENTS, ALPHABET_SIZE, SAX_STEP
    )
    sax_features.to_csv(OUTPUT_DIR / "sax_feature_matrix.csv")
    print(f"SAX feature matrix shape: {sax_features.shape}")
 
    sax_elbow = elbow_inertias(sax_features)
    plot_elbow(sax_elbow, "SAX Elbow Method", OUTPUT_DIR / "fig_sax_elbow.png")
 
    sax_km_labels, sax_km_sil, sax_km_dbi = run_kmeans(sax_features)
    sax_hc_labels, sax_hc_sil, sax_hc_dbi, sax_Z = run_hierarchical(sax_features)
 
    plot_clusters_by_series(series_dict, sax_km_labels, "SAX K-Means Clustering",
                             OUTPUT_DIR / "fig_sax_kmeans_clusters.png")
    plot_clusters_by_series(series_dict, sax_hc_labels, "SAX Hierarchical Clustering",
                             OUTPUT_DIR / "fig_sax_hierarchical_clusters.png")
    plot_dendrogram(sax_Z, list(sax_features.index), "SAX Dendrogram (Ward)",
                     OUTPUT_DIR / "fig_sax_dendrogram.png")
 
    print(f"SAX KMeans   -> Silhouette={sax_km_sil:.3f}, DBI={sax_km_dbi:.3f}")
    print(f"SAX Hierarch -> Silhouette={sax_hc_sil:.3f}, DBI={sax_hc_dbi:.3f}")
 
    # ---- 4. eSAX ----
    print("\n" + "=" * 70)
    print("STEP 4: eSAX (extended SAX: mean/max/min per segment)")
    print("=" * 70)
    esax_features, esax_words = build_esax_feature_matrix(
        series_dict, WINDOW_SIZE, N_SEGMENTS, ALPHABET_SIZE, SAX_STEP
    )
    esax_features.to_csv(OUTPUT_DIR / "esax_feature_matrix.csv")
    print(f"eSAX feature matrix shape: {esax_features.shape}")
 
    esax_elbow = elbow_inertias(esax_features)
    plot_elbow(esax_elbow, "eSAX Elbow Method", OUTPUT_DIR / "fig_esax_elbow.png")
 
    esax_km_labels, esax_km_sil, esax_km_dbi = run_kmeans(esax_features)
    esax_hc_labels, esax_hc_sil, esax_hc_dbi, esax_Z = run_hierarchical(esax_features)
 
    plot_clusters_by_series(series_dict, esax_km_labels, "eSAX K-Means Clustering",
                             OUTPUT_DIR / "fig_esax_kmeans_clusters.png")
    plot_clusters_by_series(series_dict, esax_hc_labels, "eSAX Hierarchical Clustering",
                             OUTPUT_DIR / "fig_esax_hierarchical_clusters.png")
    plot_dendrogram(esax_Z, list(esax_features.index), "eSAX Dendrogram (Ward)",
                     OUTPUT_DIR / "fig_esax_dendrogram.png")
 
    print(f"eSAX KMeans   -> Silhouette={esax_km_sil:.3f}, DBI={esax_km_dbi:.3f}")
    print(f"eSAX Hierarch -> Silhouette={esax_hc_sil:.3f}, DBI={esax_hc_dbi:.3f}")
 
    # ---- 5. TDA ----
    print("\n" + "=" * 70)
    print("STEP 5: TDA (Vietoris-Rips persistent homology + landscapes)")
    print("=" * 70)
    tda_h1, tda_h01, tda_diagrams = build_tda_features(
        series_dict, EMBED_DIM, EMBED_TAU, MAX_HOMOLOGY_DIM,
        N_LANDSCAPE_POINTS, N_LANDSCAPE_LAYERS
    )
    tda_h1.to_csv(OUTPUT_DIR / "tda_h1_feature_matrix.csv")
    tda_h01.to_csv(OUTPUT_DIR / "tda_h0h1_feature_matrix.csv")
    print(f"TDA H1 feature matrix shape: {tda_h1.shape}")
    print(f"TDA H0+H1 feature matrix shape: {tda_h01.shape}")
 
    # 5a. KMeans on H1-only landscapes (paper's first TDA strategy)
    tda_km_labels, tda_km_sil, tda_km_dbi = run_kmeans(tda_h1)
    plot_clusters_by_series(series_dict, tda_km_labels, "TDA K-Means Clustering (H1 loops)",
                             OUTPUT_DIR / "fig_tda_kmeans_clusters.png")
    print(f"TDA KMeans (H1)        -> Silhouette={tda_km_sil:.3f}, DBI={tda_km_dbi:.3f}")
 
    # 5b. Hierarchical on combined H0+H1 landscapes (paper's second TDA strategy)
    tda_hc_labels, tda_hc_sil, tda_hc_dbi, tda_Z = run_hierarchical(tda_h01)
    plot_clusters_by_series(series_dict, tda_hc_labels,
                             "TDA Hierarchical Clustering (H0+H1)",
                             OUTPUT_DIR / "fig_tda_hierarchical_clusters.png")
    plot_dendrogram(tda_Z, list(tda_h01.index), "TDA Dendrogram (Ward, H0+H1)",
                     OUTPUT_DIR / "fig_tda_dendrogram.png")
    print(f"TDA Hierarchical (H0+H1) -> Silhouette={tda_hc_sil:.3f}, DBI={tda_hc_dbi:.3f}")
 
    # ---- 6. Summary comparison table (paper Table 6) ----
    print("\n" + "=" * 70)
    print("STEP 6: Final comparison table")
    print("=" * 70)
    comparison = pd.DataFrame({
        "SAX": {
            "Silhouette (KMeans)": sax_km_sil, "DBI (KMeans)": sax_km_dbi,
            "Silhouette (Hierarchical)": sax_hc_sil, "DBI (Hierarchical)": sax_hc_dbi,
        },
        "eSAX": {
            "Silhouette (KMeans)": esax_km_sil, "DBI (KMeans)": esax_km_dbi,
            "Silhouette (Hierarchical)": esax_hc_sil, "DBI (Hierarchical)": esax_hc_dbi,
        },
        "TDA": {
            "Silhouette (KMeans)": tda_km_sil, "DBI (KMeans)": tda_km_dbi,
            "Silhouette (Hierarchical)": tda_hc_sil, "DBI (Hierarchical)": tda_hc_dbi,
        },
    })
    comparison.to_csv(OUTPUT_DIR / "table6_clustering_comparison.csv")
    print(comparison.to_string())
 
    # ---- 7. Save cluster assignments ----
    all_labels = pd.DataFrame({
        "SAX_KMeans": sax_km_labels,
        "SAX_Hierarchical": sax_hc_labels,
        "eSAX_KMeans": esax_km_labels,
        "eSAX_Hierarchical": esax_hc_labels,
        "TDA_KMeans_H1": tda_km_labels,
        "TDA_Hierarchical_H0H1": tda_hc_labels,
    })
    all_labels.to_csv(OUTPUT_DIR / "all_cluster_assignments.csv")
 
    print(f"\nAll outputs written to: {OUTPUT_DIR}")
 
 
if __name__ == "__main__":
    main()

STEP 1: Loading data
[load_all_series] Loaded 150 term series.

STEP 2: Kolmogorov-Smirnov normality test
     Keyword  KS Statistic       p-value Reject H0 at 5%
   economize      0.526254  0.000000e+00             Yes
unprofitable      0.522817  0.000000e+00             Yes
     betroth      0.518035  0.000000e+00             Yes
backwardness      0.514926  0.000000e+00             Yes
   betrothal      0.503051  0.000000e+00             Yes
    squander      0.373306 3.571425e-202             Yes
       steal      0.337678 1.796172e-164             Yes
       endow      0.329068 5.369681e-156             Yes
 breadwinner      0.325330 2.160575e-152             Yes
      hustle      0.316178 9.252016e-144             Yes

STEP 3: SAX (sliding-window symbolic aggregate approximation)
SAX feature matrix shape: (150, 12)
SAX KMeans   -> Silhouette=0.276, DBI=1.014
SAX Hierarch -> Silhouette=0.262, DBI=0.926

STEP 4: eSAX (extended SAX: mean/max/min per segment)
eSAX feature matrix shape